##### Support Request Generator Stream

This notebook creates a scheduled job to generate synthetic support requests from delivered orders

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.jobs as j
import os

w = WorkspaceClient()

CATALOG = dbutils.widgets.get("CATALOG")

notebook_abs_path = os.path.abspath("../jobs/support_request_generator")
notebook_dbx_path = notebook_abs_path.replace(
    os.environ.get("DATABRICKS_WORKSPACE_ROOT", "/Workspace"),
    "/Workspace"
)

job_name = f"Support Request Generator ({CATALOG})"

task_def = [
    j.Task(
        task_key="support_request_generator",
        notebook_task=j.NotebookTask(
            notebook_path=notebook_dbx_path,
            base_parameters={
                "CATALOG": CATALOG,
                "SUPPORT_RATE": dbutils.widgets.get("SUPPORT_RATE"),
                "LLM_MODEL": dbutils.widgets.get("LLM_MODEL"),
            },
        )
    )
]
schedule_def = j.CronSchedule(
    quartz_cron_expression="0 0/10 * * * ?",
    timezone_id="UTC",
    pause_status=j.PauseStatus.UNPAUSED,
)

existing = [jb for jb in w.jobs.list(name=job_name) if jb.settings.name == job_name]
if existing:
    job_id = existing[0].job_id
    w.jobs.reset(job_id=job_id, new_settings=j.JobSettings(
        name=job_name, tasks=task_def, schedule=schedule_def,
    ))
    print(f"\u267b\ufe0f Updated existing job_id={job_id}")
else:
    job = w.jobs.create(name=job_name, tasks=task_def, schedule=schedule_def)
    job_id = job.job_id
    import sys
    sys.path.append('../utils')
    from uc_state import add
    add(CATALOG, "jobs", job)
    print(f"\u2705 Created job_id={job_id}")

w.jobs.run_now(job_id=job_id)
print(f"\U0001f680 Started run of {job_name}")